In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
Current GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [ ]:
# Train basic gameplay model
# Stage 1 - SelfplayEnv with Random opponent

from DeepLearning.PPO import MaskablePPO
from DeepLearning.Thesis.Environments.Setup import SetupAgentCities
from DeepLearning.Thesis.DeepLearning.Thesis.Environments.SelfPlay import SelfPlayDense
from DeepLearning.GetActionMask import getActionMask, getActionMaskTrading
from DeepLearning.Thesis.Setup.getActionMaskSetup import getSetupActionMask
from DeepLearning.GetObservation import getObservation
import os
from DeepLearning.PPO import MaskablePPO

#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = SelfPlayDense()
env.selfPlay = False # for random opponents
actionMask = getActionMask
observation = getObservation

os.environ["UPDATE_MODELS_DIST"] = "False"
netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_Try2_GameplayStage_1M_SelfPlay"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

# model = MaskablePPO.load("DeepLearning/Thesis/Setup/Models/SetupRandom/model_332400_5.zip", env=env)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=3_000_000, tb_log_name=saveName, reset_num_timesteps=False)

In [3]:
model.save(savePath)
print(savePath)

DeepLearning/Models/ZKA_model/ZKA_Try2_GameplayStage_1M_SelfPlay


In [ ]:
 # Stage 2 - TurnLimitDense + AgainstRandom

from DeepLearning.Thesis.DeepLearning.Thesis.Environments.TurnLimitDense import TurnLimitDense
from Agents.AgentRandom2 import AgentRandom2
from DeepLearning.GetActionMask import getActionMask, getActionMaskTrading
from DeepLearning.GetObservation import getObservation
from DeepLearning.PPO import MaskablePPO
import os
os.environ["TURN_LIMIT"] = "30"
os.environ["UPDATE_MODELS_UNIFORM"] = "False"
os.environ["UPDATE_MODELS_DIST"] = "False"

#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = TurnLimitDense(players=[
    AgentRandom2("P0", 0),
    AgentRandom2("P1", 1),
    AgentRandom2("P2", 2),
    AgentRandom2("P3", 3)
])
actionMask = getActionMask
observation = getObservation

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_Try2_GameplayStage_+3M_Turnlimit"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

#model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

model = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_Try2_GameplayStage_3M_AgainstRandom.zip", env=env)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=3_000_000, tb_log_name=saveName, reset_num_timesteps=False)

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Policy device: cuda:0
Logging to ./tensorboard_logs_thesis/ZKA_Try2_GameplayStage_+3M_Turnlimit_0


D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\utils.py:168: UserWarning: get_schedule_fn() is deprecated, please use FloatSchedule() instead
  warnings.warn("get_schedule_fn() is deprecated, please use FloatSchedule() instead")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 64.8     |
|    ep_rew_mean     | 153      |
| time/              |          |
|    fps             | 221      |
|    iterations      | 1        |
|    time_elapsed    | 9        |
|    total_timesteps | 3002368  |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 63.7         |
|    ep_rew_mean          | 135          |
| time/                   |              |
|    fps                  | 199          |
|    iterations           | 2            |
|    time_elapsed         | 20           |
|    total_timesteps      | 3004416      |
| train/                  |              |
|    approx_kl            | 0.0061505046 |
|    clip_fraction        | 0.0456       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.791       |
|    explained_variance   | 0.214        |
|    learning_r

In [3]:
model.save(savePath)
print(savePath)

DeepLearning/Models/ZKA_model/ZKA_Try2_GameplayStage_+3M_Turnlimit


In [2]:
 # Stage 3 - Selfplay

from DeepLearning.Thesis.DeepLearning.Thesis.Environments.SelfPlay import SelfPlayBase
from Agents.AgentRandom2 import AgentRandom2
from DeepLearning.GetActionMask import getActionMask, getActionMaskTrading
from DeepLearning.GetObservation import getObservation
from DeepLearning.PPO import MaskablePPO
import os



#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = SelfPlayBase()
env.selfPlay = True
env.denseRewards = False
env.bankTradeRewards = False

actionMask = getActionMask
observation = getObservation

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_Try2_GameplayStage_6M_Selfplay"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

#model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

model = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_Try2_GameplayStage_3M_AgainstRandom.zip", env=env)
print(model.observation_space.shape)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=3_000_000, tb_log_name=saveName, reset_num_timesteps=False)

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Policy device: cuda:0
Logging to ./tensorboard_logs_thesis/ZKA_Try2_GameplayStage_6M_Selfplay_0


D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\utils.py:168: UserWarning: get_schedule_fn() is deprecated, please use FloatSchedule() instead
  warnings.warn("get_schedule_fn() is deprecated, please use FloatSchedule() instead")


CheckingWinRate(Distribution): 0.28
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 64.2     |
|    ep_rew_mean     | 117      |
| time/              |          |
|    fps             | 221      |
|    iterations      | 1        |
|    time_elapsed    | 9        |
|    total_timesteps | 3002368  |
---------------------------------
CheckingWinRate(Distribution): 0.61
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 61.7         |
|    ep_rew_mean          | 58.7         |
| time/                   |              |
|    fps                  | 198          |
|    iterations           | 2            |
|    time_elapsed         | 20           |
|    total_timesteps      | 3004416      |
| train/                  |              |
|    approx_kl            | 0.0075335046 |
|    clip_fraction        | 0.0695       |
|    clip_range           | 0.2          |
|    entropy_loss         | -

KeyboardInterrupt: 

In [3]:
model.save(savePath)
print(savePath)

DeepLearning/Models/ZKA_model/ZKA_Try2_GameplayStage_6M_Selfplay
